# 03 - Probes: seven pre-registered attempts to validate the vectors, and what survived

**What this notebook is for.** The second research question: do the emotion vectors *function*,
detecting implicit emotional content in scenarios the way the paper demonstrates? Seven
experiments, every prediction registered in TREE.md before scoring, answer: no configuration
ever passed the registered bar, on either model, under any repair. What survives is coarse
valence signal. The bench notebooks in `archive/` hold every run in full.

**Key concepts.**
- *Battery*: 12 scenarios, each implicitly evoking one emotion (the paper's Table 2, verbatim),
  plus 12 fresh held-out scenarios we wrote before any scoring, to catch selection effects.
- *Pass bar (registered)*: the target emotion ranks in the top 3 of 12 probes for at least
  8 of 12 scenarios, on BOTH batteries.
- *Readout*: which token positions represent the scenario (last token, mean of all, mean of
  content tokens).
- *Neutral projection*: the paper's confound-removal step, PCA on neutral-text activations
  with the top components projected out of the probes.

**Index.**
1. The battery sweep: gemma-4-31b-it (formats x layers x readouts), then the base arm
2. Numerical intensity: the paper's Figure 3 (the Tylenol plot), instruct vs base
3. Probe data from the model itself (self stories, dialogues)
4. The scale test: 16 to 256 stories per emotion
5. Probe-direction convergence (why scale is exonerated)
6. Neutral projection: helps, does not rescue
7. Verdict and the standing explanation

**The arc in one paragraph.** The base model failed the battery (archive 03-05, reproduced at the end of
section 1); the instruct model with its chat template failed it (section 1); probes rebuilt from the model's own
stories and dialogues failed it (section 3); scaling the corpus 16x failed to move scores
while making probes provably stable (sections 4-5); the paper's own confound projection
improved scores by about one scenario and failed to clear the bar (section 6). What does
track is numerical-intensity direction (section 2): the instruct model moves all 11
registered directions the right way, the base model 7 of 11. Geometry
replicates; probe function does not.

## 1. The battery sweep: instruct model, then base

In [1]:
# this cell loads -it probes and battery activations, then scores every sweep cell
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from emotion_vectors.probe_prompts import SCENARIOS
from emotion_vectors.scoring import score_battery

ROOT = Path("..")
bundle = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
emotions, layers, means = (
    list(map(str, bundle["emotions"])),
    list(bundle["layers"]),
    bundle["means"].astype(np.float32),
)
sweep = np.load(ROOT / "results/probe_sweep_it/activations.npz", allow_pickle=True)
formats = list(map(str, sweep["formats"]))
prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep_it/prompts.jsonl")]
READOUTS = ["last", "mean_all", "mean_content"]
assert list(sweep["layers"]) == layers
print("formats collected:", formats)

probes_by_layer_raw = means  # centering and norming live in scoring.score_battery
batteries = {
    k: [i for i, p in enumerate(prompts) if p["kind"] == k] for k in ("scenario", "heldout")
}
probe_order = [t for _, t, _ in SCENARIOS]
probe_idx = [emotions.index(t) for t in probe_order]


def top3_count(fmt, kind, readout, layer_pos):
    """Thin wrapper binding sweep data to the promoted scorer."""
    acts = sweep[f"{fmt}_{readout}"].astype(np.float32)
    A = np.stack([acts[i, layer_pos] for i in batteries[kind]])
    count, M = score_battery(
        probes_by_layer_raw[probe_idx, layer_pos, :],
        A,
        center_pool=probes_by_layer_raw[:, layer_pos, :],  # sweep convention: center on all 171
    )
    return count, M


grids = {
    (fmt, kind): np.array(
        [[top3_count(fmt, kind, r, lp)[0] for r in READOUTS] for lp in range(len(layers))]
    )
    for fmt in formats
    for kind in ("scenario", "heldout")
}
print("grids:", {k: v.max() for k, v in grids.items()})

formats collected: ['plain', 'chat']


grids: {('plain', 'scenario'): np.int64(4), ('plain', 'heldout'): np.int64(4), ('chat', 'scenario'): np.int64(6), ('chat', 'heldout'): np.int64(4)}


In [2]:
# this cell draws the four sweep grids with per-cell scores (bold at the pass bar of 8)
titles = [
    f"{fmt}: {'paper (selection)' if kind == 'scenario' else 'held-out (confirmation)'}"
    for fmt in formats
    for kind in ("scenario", "heldout")
]
fig = make_subplots(
    rows=len(formats),
    cols=2,
    subplot_titles=titles,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    vertical_spacing=0.10,
)
for fi, fmt in enumerate(formats):
    for ki, kind in enumerate(("scenario", "heldout")):
        G = grids[(fmt, kind)]
        fig.add_trace(
            go.Heatmap(
                z=G,
                colorscale="Viridis",
                zmin=0,
                zmax=12,
                showscale=(fi == 0 and ki == 0),
                colorbar=dict(title="scenarios with target in top-3 (threshold 8)"),
            ),
            row=fi + 1,
            col=ki + 1,
        )
        for lp in range(len(layers)):
            for rp in range(len(READOUTS)):
                v = int(G[lp, rp])
                fig.add_annotation(
                    x=rp,
                    y=lp,
                    text=f"<b>{v}</b>" if v >= 8 else str(v),
                    showarrow=False,
                    font=dict(size=9, color="white" if v < 7 else "black"),
                    row=fi + 1,
                    col=ki + 1,
                )
        fig.update_xaxes(
            tickvals=list(range(len(READOUTS))),
            ticktext=READOUTS,
            tickangle=20,
            title_text="readout",
            row=fi + 1,
            col=ki + 1,
        )
        fig.update_yaxes(autorange="reversed", row=fi + 1, col=ki + 1)
    fig.update_yaxes(
        tickvals=list(range(len(layers))),
        ticktext=[str(l) for l in layers],
        title_text="layer",
        row=fi + 1,
        col=1,
    )
fig.update_layout(
    title="Q1.H2.E4: the faithful battery, gemma-4-31b-it probes and formats<br><sup>our extension (no source figure): robustness sweep for Anthropic Figure 2</sup>",
    height=520 * len(formats),
    width=900,
)
fig.add_annotation(
    text="no cell reaches the registered bar of 8",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.06,
    showarrow=False,
    font=dict(size=13),
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Each cell is one layer-readout combination; the number counts scenarios (of 12) whose target emotion ranked top-3. Chance is about 3; the registered bar is 8, and a combination only counts if it clears 8 on the selection panel AND its held-out twin. Bright-left-dim-right pairs are the selection mirages the held-out battery exists to catch.

</details>

In [3]:
# this cell ranks the top combinations and applies the registered rule
flat = [
    (grids[(f, "scenario")][lp, rp], grids[(f, "heldout")][lp, rp], f, layers[lp], r)
    for f in formats
    for lp in range(len(layers))
    for rp, r in enumerate(READOUTS)
]
flat.sort(reverse=True)
print("top combinations (paper, heldout, format, layer, readout):")
for row in flat[:10]:
    print(
        f"  paper {row[0]:2d}/12  heldout {row[1]:2d}/12  {row[2]:5s}  layer {row[3]:2d}  {row[4]}"
    )
confirmed = [row for row in flat if row[0] >= 8 and row[1] >= 8]
print(
    f"\nregistered rule — combinations passing BOTH batteries at >=8/12: "
    f"{confirmed if confirmed else 'NONE'}"
)

top combinations (paper, heldout, format, layer, readout):
  paper  6/12  heldout  4/12  chat   layer 57  last
  paper  4/12  heldout  4/12  plain  layer 48  last
  paper  4/12  heldout  4/12  plain  layer 45  mean_all
  paper  4/12  heldout  4/12  plain  layer 27  mean_content
  paper  4/12  heldout  4/12  plain  layer 27  mean_all
  paper  4/12  heldout  4/12  chat   layer 51  mean_content
  paper  4/12  heldout  4/12  chat   layer 51  mean_all
  paper  4/12  heldout  3/12  plain  layer 54  mean_all
  paper  4/12  heldout  3/12  plain  layer 45  last
  paper  4/12  heldout  2/12  plain  layer 57  mean_content

registered rule — combinations passing BOTH batteries at >=8/12: NONE


In [4]:
# this cell scores the base-model sweep the same way (plain format only) and draws its grid
base_bundle = np.load(ROOT / "results/emotion_vectors/emotion_means.npz", allow_pickle=True)
base_emotions = list(map(str, base_bundle["emotions"]))
base_means = base_bundle["means"].astype(np.float32)
base_sweep = np.load(ROOT / "results/probe_sweep/activations.npz", allow_pickle=True)
base_formats = list(map(str, base_sweep["formats"]))
base_prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep/prompts.jsonl")]
base_batteries = {
    k: [i for i, p in enumerate(base_prompts) if p["kind"] == k] for k in ("scenario", "heldout")
}
base_probe_idx = [base_emotions.index(t) for t in probe_order]
assert list(base_sweep["layers"]) == layers
assert base_formats == ["plain"], "base model has no chat template; plain format only"


def base_top3(kind, readout, lp):
    """Same scorer, base probes on base activations, centered on the base 171."""
    acts = base_sweep[f"plain_{readout}"].astype(np.float32)
    A = np.stack([acts[i, lp] for i in base_batteries[kind]])
    count, M = score_battery(
        base_means[base_probe_idx, lp, :],
        A,
        center_pool=base_means[:, lp, :],  # sweep convention: center on all 171
    )
    return count


base_grids = {
    kind: np.array([[base_top3(kind, r, lp) for r in READOUTS] for lp in range(len(layers))])
    for kind in ("scenario", "heldout")
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["plain: paper (selection)", "plain: held-out (confirmation)"],
    shared_yaxes=True,
    horizontal_spacing=0.08,
)
for ki, kind in enumerate(("scenario", "heldout")):
    G = base_grids[kind]
    fig.add_trace(
        go.Heatmap(
            z=G,
            colorscale="Viridis",
            zmin=0,
            zmax=12,
            showscale=(ki == 0),
            colorbar=dict(title="scenarios with target in top-3 (threshold 8)"),
        ),
        row=1,
        col=ki + 1,
    )
    for lp in range(len(layers)):
        for rp in range(len(READOUTS)):
            v = int(G[lp, rp])
            fig.add_annotation(
                x=rp,
                y=lp,
                text=f"<b>{v}</b>" if v >= 8 else str(v),
                showarrow=False,
                font=dict(size=9, color="white" if v < 7 else "black"),
                row=1,
                col=ki + 1,
            )
    fig.update_xaxes(
        tickvals=list(range(len(READOUTS))),
        ticktext=READOUTS,
        tickangle=20,
        title_text="readout",
        row=1,
        col=ki + 1,
    )
    fig.update_yaxes(autorange="reversed", row=1, col=ki + 1)
fig.update_yaxes(
    tickvals=list(range(len(layers))),
    ticktext=[str(l) for l in layers],
    title_text="layer",
    row=1,
    col=1,
)
fig.update_layout(
    title="The faithful battery, gemma-4-31b (base) probes, plain format only<br><sup>our extension (no source figure): robustness sweep for Anthropic Figure 2</sup>",
    height=560,
    width=900,
)
fig.add_annotation(
    text="no cell reaches the registered bar of 8",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.10,
    showarrow=False,
    font=dict(size=13),
)
fig.show()

base_flat = [
    (base_grids["scenario"][lp, rp], base_grids["heldout"][lp, rp], "plain", int(layers[lp]), r)
    for lp in range(len(layers))
    for rp, r in enumerate(READOUTS)
]
base_flat.sort(reverse=True)
print("top combinations, gemma-4-31b base (paper, heldout, format, layer, readout):")
for row in base_flat[:5]:
    print(
        f"  paper {row[0]:2d}/12  heldout {row[1]:2d}/12  {row[2]:5s}  layer {row[3]:2d}  {row[4]}"
    )
best = base_flat[0]
assert (int(best[0]), int(best[1]), best[3], best[4]) == (7, 4, 57, "last"), (
    f"base sweep no longer reproduces the archived best cell (archive/04_probe_sweep): {best}"
)
print(
    "reproduction check vs archive/04_probe_sweep — "
    "best cell 7/12 paper, 4/12 heldout at layer 57, last: OK"
)
base_confirmed = [row for row in base_flat if row[0] >= 8 and row[1] >= 8]
print(
    f"\nregistered rule (gemma-4-31b base) — combinations passing BOTH batteries at >=8/12: "
    f"{base_confirmed if base_confirmed else 'NONE'}"
)

top combinations, gemma-4-31b base (paper, heldout, format, layer, readout):
  paper  7/12  heldout  4/12  plain  layer 57  last
  paper  5/12  heldout  5/12  plain  layer 57  mean_content
  paper  5/12  heldout  4/12  plain  layer 57  mean_all
  paper  4/12  heldout  4/12  plain  layer 33  mean_content
  paper  4/12  heldout  4/12  plain  layer 33  mean_all
reproduction check vs archive/04_probe_sweep — best cell 7/12 paper, 4/12 heldout at layer 57, last: OK

registered rule (gemma-4-31b base) — combinations passing BOTH batteries at >=8/12: NONE


<details><summary><b>How to read this figure</b></summary>

Same grid as above, computed for the base model (`gemma-4-31b`) from its own probes and battery activations. The base model has no chat template, so only the plain format exists — one panel pair instead of two. The instruct model above is the arm we care about; the base arm is the comparison showing the failure predates chat formatting. The base model's best cell (7/12 paper, 4/12 held-out, at layer 57, last-token readout) exactly reproduces the archived bench run (`archive/04_probe_sweep.ipynb`) and still fails the registered bar on the held-out battery.

</details>

## 2. Numerical intensity: the paper's Figure 3 (the Tylenol plot), instruct vs base

Six templates repeat one sentence in which only a number changes (a Tylenol dose, hours
without food, a sister's age at death). If probes track meaning rather than digits, the
cosines must move with what the number implies. This is E1's second registered prediction
(P2), shown here for both models.

In [5]:
# this cell scores the six numerical-intensity templates for both models and draws the 3x2 grid
from scipy.stats import spearmanr

from emotion_vectors.probe_prompts import TRACKED_PROBES
from emotion_vectors.scoring import battery_matrix

NI_LAYER = 33  # E1's registered layer; readout: last token (E1's registered readout)
PROBE_COLORS = {"afraid": "#d62728", "calm": "#1f77b4", "happy": "#2ca02c", "sad": "#ff7f0e"}
REGISTERED_DIRECTIONS = [  # TREE.md Q1.H2.E1 (P2): the 11 named (template, probe, sign) pairs
    ("tylenol", "afraid", +1),
    ("tylenol", "calm", -1),
    ("fasting", "afraid", +1),
    ("sister", "sad", -1),
    ("sister", "calm", +1),
    ("sister", "happy", +1),
    ("dog_missing", "sad", +1),
    ("runway", "afraid", -1),
    ("runway", "calm", +1),
    ("exam", "happy", +1),
    ("exam", "afraid", -1),
]

ni_models = {}  # label -> {template name: (xs, 4xN cosine matrix)}
for label, sweep_dir, means_path, fmt in (
    ("gemma-4-31b-it", "results/probe_sweep_it", "results/emotion_vectors_it_means.npz", "chat"),
    (
        "gemma-4-31b (base)",
        "results/probe_sweep",
        "results/emotion_vectors/emotion_means.npz",
        "plain",
    ),
):
    ni_bundle = np.load(ROOT / means_path, allow_pickle=True)
    ni_emotions = list(map(str, ni_bundle["emotions"]))
    ni_means = ni_bundle["means"].astype(np.float32)
    lp = [int(l) for l in ni_bundle["layers"]].index(NI_LAYER)
    ni_acts = np.load(ROOT / f"{sweep_dir}/activations.npz", allow_pickle=True)[f"{fmt}_last"]
    ni_prompts = [json.loads(l) for l in open(ROOT / f"{sweep_dir}/prompts.jsonl")]
    ni_pidx = [ni_emotions.index(e) for e in TRACKED_PROBES]
    by_name = {}
    for i, p in enumerate(ni_prompts):
        if p["kind"] == "template":
            by_name.setdefault(p["name"], []).append((p["x"], i, p))
    curves = {}
    for name, pts in by_name.items():
        pts.sort()
        A = np.stack([ni_acts[i, lp].astype(np.float32) for _, i, _ in pts])
        M = battery_matrix(ni_means[ni_pidx, lp, :], A, center_pool=ni_means[:, lp, :])
        curves[name] = ([x for x, _, _ in pts], M, pts[0][2])
    ni_models[label] = curves

TEMPLATES = ["tylenol", "fasting", "sister", "dog_missing", "runway", "exam"]
panel_titles = []
for name in TEMPLATES:
    xs, _, row0 = ni_models["gemma-4-31b-it"][name]
    panel_titles.append(row0["text"].replace(str(xs[0]), "{X}", 1))
fig = make_subplots(
    rows=3, cols=2, subplot_titles=panel_titles, vertical_spacing=0.12, horizontal_spacing=0.09
)
for a in fig.layout.annotations:  # only the six panel titles exist at this point
    a.font = dict(size=10)
for t, name in enumerate(TEMPLATES):
    r, c = t // 2 + 1, t % 2 + 1
    for label, dash, width in (("gemma-4-31b-it", None, 2.5), ("gemma-4-31b (base)", "dash", 1.5)):
        xs, M, row0 = ni_models[label][name]
        for j, probe in enumerate(TRACKED_PROBES):
            fig.add_trace(
                go.Scatter(
                    x=list(range(len(xs))),
                    y=M[j],
                    mode="lines+markers",
                    name=f"{probe} ({'instruct' if 'it' in label else 'base'})",
                    legendgroup=f"{probe}-{label}",
                    showlegend=(t == 0),
                    line=dict(color=PROBE_COLORS[probe], dash=dash, width=width),
                    marker=dict(size=5),
                ),
                row=r,
                col=c,
            )
    fig.add_hline(y=0, line_color="#888", line_width=1, row=r, col=c)
    fig.update_xaxes(
        tickvals=list(range(len(xs))),
        ticktext=[str(x) for x in xs],
        title_text=row0["axis"],
        title_font=dict(size=10),
        row=r,
        col=c,
    )
    fig.update_yaxes(title_text="cosine" if c == 1 else None, row=r, col=c)

xs, M_it, _ = ni_models["gemma-4-31b-it"]["tylenol"]
rho_a = spearmanr(xs, M_it[TRACKED_PROBES.index("afraid")]).statistic
rho_c = spearmanr(xs, M_it[TRACKED_PROBES.index("calm")]).statistic
fig.add_annotation(
    text=(
        f"instruct: afraid {'rises' if rho_a > 0 else 'falls'} (rho {rho_a:+.2f}), "
        f"calm {'rises' if rho_c > 0 else 'falls'} (rho {rho_c:+.2f}) with dose"
    ),
    x=2.5,
    y=float(M_it.mean()),
    showarrow=False,
    font=dict(size=9),
    row=1,
    col=1,
)
fig.update_layout(
    title="Numerical-intensity templates (paper Figure 3), layer 33 last token: "
    "gemma-4-31b-it (solid, chat) vs base (dashed, plain)<br><sup>maps to: Anthropic Figure 3</sup>",
    height=980,
    width=1050,
)
fig.show()

for label in ("gemma-4-31b-it", "gemma-4-31b (base)"):
    correct = 0
    print(f"{label} — Spearman sign per registered (template, probe) direction:")
    for name, probe, sign in REGISTERED_DIRECTIONS:
        xs, M, _ = ni_models[label][name]
        rho = spearmanr(xs, M[TRACKED_PROBES.index(probe)]).statistic
        ok = np.sign(rho) == sign
        correct += int(ok)
        print(
            f"  {name:12s} {probe:7s} registered {'+' if sign > 0 else '-'}  "
            f"rho {rho:+.3f}  {'correct' if ok else 'WRONG'}"
        )
    print(f"{label}: {correct}/11 registered directions correct\n")
    if label == "gemma-4-31b (base)":
        assert correct == 7, f"base arm should reproduce E1's 7/11 (got {correct}/11)"
        print("reproduction check vs Q1.H2.E1 (plain last, layer 33) — 7/11: OK")

gemma-4-31b-it — Spearman sign per registered (template, probe) direction:
  tylenol      afraid  registered +  rho +0.943  correct
  tylenol      calm    registered -  rho -1.000  correct
  fasting      afraid  registered +  rho +0.857  correct
  sister       sad     registered -  rho -0.829  correct
  sister       calm    registered +  rho +1.000  correct
  sister       happy   registered +  rho +0.600  correct
  dog_missing  sad     registered +  rho +1.000  correct
  runway       afraid  registered -  rho -0.943  correct
  runway       calm    registered +  rho +0.943  correct
  exam         happy   registered +  rho +1.000  correct
  exam         afraid  registered -  rho -0.600  correct
gemma-4-31b-it: 11/11 registered directions correct

gemma-4-31b (base) — Spearman sign per registered (template, probe) direction:
  tylenol      afraid  registered +  rho +0.771  correct
  tylenol      calm    registered -  rho -1.000  correct
  fasting      afraid  registered +  rho -0.857  WRO

<details><summary><b>How to read this figure</b></summary>

Each panel is one template: the same sentence, only the number changes (x axis, categorical positions labeled with the actual values). The y axis is the cosine between the scenario's activation and each of the four tracked probes (afraid red, calm blue, happy green, sad orange), measured at E1's registered readout and layer — the last token at layer 33; the sweep npz files hold every other layer and readout choice. Solid lines are `gemma-4-31b-it` under its chat template; dashed lines are the base model in plain format (the base model has no chat template). Curves moving with what the number *means* — more Tylenol raising afraid and lowering calm, a longer-lived sister lowering sad, more runway lowering afraid — is semantic tracking; moving with the digits alone is not. The registered scoring (TREE.md Q1.H2.E1, prediction P2) is the Spearman sign of the 11 named (template, probe) pairs, printed below the figure: the instruct model gets 11/11 correct, the base model 7/11 — and the base count reproduces E1's archived result exactly (E1 measured plain-format last-token at layer 33, the same cells recomputed here; fasting and dog-missing invert, sister is mixed). The visible slopes are small next to the between-probe offsets, which is why the registered read is the sign test, not the visual slope.



**Confound check (Q1.H2.E8, results/e8_template_diagnostic.json).** These curves must be read
in sign only, not magnitude. A registered diagnostic ran three controls on the instruct arm:
(1) *random-direction null* — random directions in probe span reach |Spearman rho| = 1 with N
so often (null 95th percentile 1.00 on 5 of 6 panels) that rho magnitude is not evidence: the
last-token activation drifts near-monotonically with N regardless of emotion content, and the
activation norm itself tracks N (|rho| 0.54-0.83 on 5 of 6 panels); (2) *amplitude* — our
per-curve cosine excursions are 0.002-0.016 against the paper's ~0.16 full range, 10-50x
smaller (the visible slopes here are plot autoscale); (3) the neutral-PC projection changes
neither. What survives is exactly the registered sign read: under the generic-drift null each
named direction is a coin flip, and 11/11 correct signs is p about 2^-11. The templates detect
a real but tiny, sign-consistent emotional tilt riding on a much larger generic drift —
consistent with every other readout on this model.

</details>

## 3. Probe data from the model itself (gemma-4-31b-it only)

*Instruct-only by design: the self-generated story and dialogue corpora exist only for `gemma-4-31b-it`. The base arm has no equivalent here (base-model dialogue probes live in `archive/05_dialogue_probes.ipynb`).*

In [6]:
# this cell loads all probe sets and compares self-gen directions to corpus directions
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from scipy.stats import rankdata

from emotion_vectors.probe_prompts import SCENARIOS
from emotion_vectors.story_store import emotion_mean, load_manifest

ROOT = Path("..")
probe_order = [t for _, t, _ in SCENARIOS]

bundle = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
it_emotions, layers, it_means = (
    list(map(str, bundle["emotions"])),
    list(bundle["layers"]),
    bundle["means"].astype(np.float32),
)
corpus12 = np.stack([it_means[it_emotions.index(e)] for e in probe_order])


def means_from(results_dir):
    rows = [
        r for r in load_manifest(results_dir / "manifest.jsonl").values() if r.get("error") is None
    ]
    by_emotion = {}
    for r in rows:
        by_emotion.setdefault(r["emotion"], []).append(r)
    return np.stack(
        [
            np.stack(
                [
                    np.load(results_dir / e.replace(" ", "_") / f"layer_{layer}_resid.npy")
                    for layer in layers
                ]
            )
            for e in probe_order
        ]
    )


selfstory12 = means_from(ROOT / "results/self_story_vectors_it")
dlg12 = means_from(ROOT / "results/dialogue_vectors_it")

sweep = np.load(ROOT / "results/probe_sweep_it/activations.npz", allow_pickle=True)
formats = list(map(str, sweep["formats"]))
prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep_it/prompts.jsonl")]
READOUTS = ["last", "mean_all", "mean_content"]
batteries = {
    k: [i for i, p in enumerate(prompts) if p["kind"] == k] for k in ("scenario", "heldout")
}

# registered comparison: how similar are self-gen probes to corpus probes? (layer 33, centered)
print("model: gemma-4-31b-it (all probe sets and battery activations)")
for name, pm in (("self-story", selfstory12), ("dialogue", dlg12)):
    lp = layers.index(33)
    a = corpus12[:, lp, :] - corpus12[:, lp, :].mean(0)
    b = pm[:, lp, :] - pm[:, lp, :].mean(0)
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    print(
        f"probe-direction cosine vs 4B-corpus probes ({name}, layer 33): "
        f"mean {cos.mean():.3f}, min {cos.min():.3f}"
    )

model: gemma-4-31b-it (all probe sets and battery activations)
probe-direction cosine vs 4B-corpus probes (self-story, layer 33): mean 0.220, min -0.027
probe-direction cosine vs 4B-corpus probes (dialogue, layer 33): mean 0.093, min -0.201


In [7]:
# this cell scores every (probe set, format, layer, readout) and applies the registered rule
def score(pm, fmt, kind, readout, lp):
    acts = sweep[f"{fmt}_{readout}"].astype(np.float32)
    P = pm[:, lp, :] - pm[:, lp, :].mean(axis=0)
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    A = np.stack([acts[i, lp] for i in batteries[kind]])
    M = P @ A.T / np.linalg.norm(A, axis=1)
    ranks = [int(12 - rankdata(M[:, j])[j]) + 1 for j in range(12)]
    return sum(r <= 3 for r in ranks), M


SETS = {"corpus": corpus12, "self-story": selfstory12, "dialogue": dlg12}
rows, confirmed = [], []
for name, pm in SETS.items():
    for fmt in formats:
        for lp, layer in enumerate(layers):
            for readout in READOUTS:
                t_p, _ = score(pm, fmt, "scenario", readout, lp)
                t_h, _ = score(pm, fmt, "heldout", readout, lp)
                rows.append((t_p, t_h, name, fmt, layer, readout))
                if t_p >= 8 and t_h >= 8:
                    confirmed.append(rows[-1])
rows.sort(reverse=True)
print("probe sets scored on gemma-4-31b-it battery activations:")
print(f"{'paper':>6s} {'heldout':>8s}  set         fmt    layer readout")
seen = {}
for r in rows:
    if seen.get(r[2], 0) < 4:
        seen[r[2]] = seen.get(r[2], 0) + 1
        print(f"  {r[0]:2d}/12    {r[1]:2d}/12  {r[2]:11s} {r[3]:6s} {r[4]:3d}  {r[5]}")
print(
    f"\nregistered rule — combinations passing BOTH batteries at >=8/12: "
    f"{confirmed if confirmed else 'NONE'}"
)

probe sets scored on gemma-4-31b-it battery activations:
 paper  heldout  set         fmt    layer readout
   7/12     6/12  self-story  chat    57  mean_content
   7/12     6/12  self-story  chat    57  mean_all
   6/12     5/12  self-story  chat    57  last
   6/12     5/12  dialogue    chat    57  mean_all
   6/12     4/12  dialogue    chat    57  last
   5/12     4/12  corpus      chat    57  last
   4/12     5/12  self-story  chat    54  last
   4/12     4/12  dialogue    plain   24  mean_all
   4/12     4/12  dialogue    chat    51  mean_content
   4/12     4/12  corpus      plain   27  mean_content
   4/12     4/12  corpus      plain   27  mean_all
   4/12     4/12  corpus      chat    48  last

registered rule — combinations passing BOTH batteries at >=8/12: NONE


<details><summary><b>How to read this output</b></summary>

Rows are probe sets scored at the registered layers; corpus = vectors from the published stories, self-story and dialogue = vectors from the model's own generations (leakage 1% and 0%). Self-generated probes give the campaign's best cells but none passes.

</details>

## 4. The scale test (gemma-4-31b-it only)

*Instruct-only by design: the 16-256-stories-per-emotion scale corpus was generated by and extracted from `gemma-4-31b-it`; no base-arm equivalent exists.*

In [8]:
# this cell loads all inputs and defines the one scoring function used throughout
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata

from emotion_vectors.analysis import project_out_neutral

ROOT = Path("..")
e6 = np.load(ROOT / "results/e6_scale_means.npz", allow_pickle=True)
MEANS = e6["means"].astype(np.float32)  # [n_bucket, emotion, layer, d_model]
N_BUCKETS = list(e6["n_buckets"])
EMOTIONS = list(map(str, e6["emotions"]))
LAYERS = list(e6["layers"])
neutral = np.load(ROOT / "results/e7_neutral_bundle.npz")["vectors"].astype(np.float32)

sweep = np.load(ROOT / "results/probe_sweep_it/activations.npz", allow_pickle=True)
FORMATS = list(map(str, sweep["formats"]))
prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep_it/prompts.jsonl")]
BATTERIES = {
    k: [i for i, p in enumerate(prompts) if p["kind"] == k] for k in ("scenario", "heldout")
}
READOUTS = ["last", "mean_all", "mean_content"]


def top3(probe_means_ed, fmt, kind, readout, lp):
    """Scenarios (of 12) whose target emotion ranks top-3 among the 12 probes."""
    acts = sweep[f"{fmt}_{readout}"].astype(np.float32)
    P = probe_means_ed - probe_means_ed.mean(axis=0)
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    A = np.stack([acts[i, lp] for i in BATTERIES[kind]])
    M = P @ A.T / np.linalg.norm(A, axis=1)
    return sum(int(12 - rankdata(M[:, j])[j]) + 1 <= 3 for j in range(12)), M


print(f"model: gemma-4-31b-it | {MEANS.shape=} | formats {FORMATS} | {len(LAYERS)} layers")

model: gemma-4-31b-it | MEANS.shape=(4, 12, 20, 5376) | formats ['plain', 'chat'] | 20 layers


In [9]:
# this cell scores the battery at every n and plots the scale curve
def best_scores(probe_means_layers):  # [emotion, layer, d_model] -> per-battery best + argmax
    best = {"scenario": 0, "heldout": 0}
    best_cfg, joint_best = None, -1
    for fmt in FORMATS:
        for lp in range(len(LAYERS)):
            for r in READOUTS:
                s, _ = top3(probe_means_layers[:, lp, :], fmt, "scenario", r, lp)
                h, _ = top3(probe_means_layers[:, lp, :], fmt, "heldout", r, lp)
                best["scenario"] = max(best["scenario"], s)
                best["heldout"] = max(best["heldout"], h)
                if min(s, h) > joint_best:
                    joint_best, best_cfg = min(s, h), (fmt, LAYERS[lp], r, s, h)
    return best, best_cfg


scale_rows = []
for ni, n in enumerate(N_BUCKETS):
    best, cfg = best_scores(MEANS[ni])
    scale_rows.append((n, best["scenario"], best["heldout"], cfg))
    print(
        f"n={n:3d}: best paper {best['scenario']}/12, best heldout {best['heldout']}/12, "
        f"best joint config {cfg}"
    )

fig = go.Figure()
fig.add_scatter(
    x=[r[0] for r in scale_rows],
    y=[r[1] for r in scale_rows],
    mode="lines+markers",
    name="paper battery (best)",
)
fig.add_scatter(
    x=[r[0] for r in scale_rows],
    y=[r[2] for r in scale_rows],
    mode="lines+markers",
    name="held-out battery (best)",
    line=dict(dash="dash"),
)
fig.add_hline(y=8, line_dash="dot", annotation_text="registered pass bar (8/12)")
fig.add_annotation(
    text="flat: more data does not buy a diagonal",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.12,
    showarrow=False,
)
fig.update_layout(
    title="E6: battery score vs probe-corpus size (gemma-4-31b-it)<br><sup>our extension (no source figure): scale test</sup>",
    xaxis_title="stories per emotion (log scale)",
    xaxis_type="log",
    yaxis_title="scenarios with target in top-3 (of 12)",
    yaxis_range=[0, 12],
    height=420,
)
fig.show()

n= 16: best paper 5/12, best heldout 6/12, best joint config ('chat', np.int64(39), 'last', 5, 5)


n= 64: best paper 5/12, best heldout 4/12, best joint config ('plain', np.int64(42), 'mean_content', 4, 4)


n=128: best paper 5/12, best heldout 5/12, best joint config ('chat', np.int64(57), 'last', 5, 5)


n=256: best paper 4/12, best heldout 5/12, best joint config ('plain', np.int64(6), 'mean_all', 4, 4)


<details><summary><b>How to read this figure</b></summary>

Solid line: best score on the paper battery at each corpus size; dashed: the held-out battery. The dotted bar is the registered pass threshold. A rising curve crossing the bar would vindicate the scale hypothesis; the observed flat curve exonerates scale.

</details>

## 5. Probe-direction convergence (gemma-4-31b-it only)

*Instruct-only by design: convergence is measured on the `gemma-4-31b-it` scale corpus from section 4; no base-arm equivalent exists.*

In [10]:
# this cell measures probe-direction convergence toward the n=256 reference
LP33 = LAYERS.index(33)
ref = MEANS[-1, :, LP33, :] - MEANS[-1, :, LP33, :].mean(axis=0)
conv = []
for ni, n in enumerate(N_BUCKETS[:-1]):
    cur = MEANS[ni, :, LP33, :] - MEANS[ni, :, LP33, :].mean(axis=0)
    cos = (cur * ref).sum(1) / (np.linalg.norm(cur, axis=1) * np.linalg.norm(ref, axis=1))
    conv.append(float(cos.mean()))
    print(
        f"n={n:3d} vs n=256: mean contrast-direction cosine {cos.mean():.3f} (min {cos.min():.3f})"
    )
fig = go.Figure(go.Scatter(x=N_BUCKETS[:-1], y=conv, mode="lines+markers"))
fig.update_layout(
    title="E6: probe-direction convergence with corpus size (gemma-4-31b-it, layer 33)<br><sup>our extension (no source figure): scale test</sup>",
    xaxis_title="stories per emotion (log scale)",
    xaxis_type="log",
    yaxis_title="cosine vs n=256 direction",
    height=380,
)
fig.show()

n= 16 vs n=256: mean contrast-direction cosine 0.944 (min 0.894)
n= 64 vs n=256: mean contrast-direction cosine 0.991 (min 0.983)
n=128 vs n=256: mean contrast-direction cosine 0.997 (min 0.993)


<details><summary><b>How to read this figure</b></summary>

For each corpus size, the cosine between that size's probe direction and the final 256-story direction, averaged over emotions. Rising to 0.997 by 128 stories means the probes are data-stable; combined with section 3's flat curve, noise cannot be what suppresses the diagonal.

</details>

## 6. Neutral projection (gemma-4-31b-it only)

*Instruct-only by design: the neutral transcripts used for the confound projection were generated by `gemma-4-31b-it`; no base-arm equivalent exists.*

In [11]:
# this cell scores projected vs unprojected probes and checks the valence-PC position
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

n256 = MEANS[-1]  # [emotion, layer, d_model]
proj_means = np.zeros_like(n256)
removed = {}
for lp in range(len(LAYERS)):
    proj_means[:, lp, :], k = project_out_neutral(n256[:, lp, :], neutral[:, lp, :])
    removed[LAYERS[lp]] = k
print("model: gemma-4-31b-it")
print("neutral PCs removed per layer (50% variance):", removed)

print(f"\n{'probes':12s} {'fmt':6s} {'layer':5s} {'readout':13s} {'paper':>6s} {'heldout':>8s}")
for name, pm in (("unprojected", n256), ("projected", proj_means)):
    for fmt in FORMATS:
        for layer in (33, 39, 57):
            lp = LAYERS.index(layer)
            for r in ("last", "mean_content"):
                s, _ = top3(pm[:, lp, :], fmt, "scenario", r, lp)
                h, _ = top3(pm[:, lp, :], fmt, "heldout", r, lp)
                if s + h >= 8 or (layer == 57 and r == "mean_content"):
                    print(f"{name:12s} {fmt:6s} {layer:5d} {r:13s} {s:4d}/12 {h:6d}/12")

best_unproj, cfg_u = best_scores(n256)
best_proj, cfg_p = best_scores(proj_means)
print(f"\nbest joint (unprojected): {cfg_u}")
print(f"best joint (projected):   {cfg_p}")
PASS = [c for c in [cfg_u, cfg_p] if c and c[3] >= 8 and c[4] >= 8]
print(f"registered rule, either probe set: {'PASS ' + str(PASS) if PASS else 'NONE pass'}")

model: gemma-4-31b-it
neutral PCs removed per layer (50% variance): {np.int64(0): 4, np.int64(3): 6, np.int64(6): 7, np.int64(9): 7, np.int64(12): 3, np.int64(15): 4, np.int64(18): 4, np.int64(21): 4, np.int64(24): 5, np.int64(27): 8, np.int64(30): 7, np.int64(33): 5, np.int64(36): 5, np.int64(39): 4, np.int64(42): 4, np.int64(45): 3, np.int64(48): 3, np.int64(51): 3, np.int64(54): 4, np.int64(57): 4}

probes       fmt    layer readout        paper  heldout


unprojected  plain     57 mean_content     3/12      3/12


unprojected  chat      57 last             4/12      5/12
unprojected  chat      57 mean_content     4/12      3/12


projected    plain     57 mean_content     4/12      4/12
projected    chat      39 last             5/12      5/12


projected    chat      39 mean_content     4/12      4/12
projected    chat      57 last             4/12      6/12
projected    chat      57 mean_content     5/12      4/12



best joint (unprojected): ('plain', np.int64(6), 'mean_all', 4, 4)
best joint (projected):   ('chat', np.int64(18), 'mean_all', 5, 5)
registered rule, either probe set: NONE pass


<details><summary><b>How to read this output</b></summary>

Battery scores with unprojected vs projected probes at the full corpus size. Projection adds roughly one scenario in several cells (best joint configuration moves from 4/4 to 5/5) and no configuration passes. The geometry effect of the same projection is in the geometry notebook, section 7.

</details>

## 7. Verdict and the standing explanation

Every methodological repair the source papers and our own analysis could motivate has been
tried under pre-registered rules: instruct model, chat formatting, all layers and readouts,
dialogue-form probes, generator-matched corpora, a 16x scale-up, and the paper's confound
projection. Best observed configuration: 7 of 12 on the paper battery with 6 of 12 held-out,
against a bar of 8 and 8.

The standing explanation (claim C3, formal graduation pending the falsify gate): a genuine
model difference from the paper's reinforcement-learning-from-human-feedback Claude models,
with one concrete anomaly to explain, the dominant non-affective component in the instruct
model that survives neutral projection. TREE.md nodes Q1.H2.E1 through E7 carry every verdict
with evidence paths.